# AlarmLab — research walkthrough

This executed notebook checks the supplied data and explains the event-level evaluation. See the methods document for the exact model definitions. It does not claim clinical or plant validation.

In [1]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd

root = Path.cwd()
if not (root / "configs").exists():
    root = root.parent
assert (root / "configs/default.json").exists(), "Open from the repo or notebooks folder."
sys.path.insert(0, str(root / "src"))
from alarmlab.io import sha256
manifest = json.loads((root / "artifacts/manifest.json").read_text())
assert sha256(root / "artifacts/synthetic_observations.csv.gz") == manifest["input_sha256"]
print("Input checksum verified.")
print("Episodes:", manifest["episodes"], "| Samples:", manifest["rows"])


Input checksum verified.
Episodes: 148 | Samples: 88800


## 1. Verify the independent units

A run is the unit of the synthetic split. The observation count is much larger than the number of independent runs.

In [2]:
data = pd.read_csv(root / "artifacts/synthetic_observations.csv.gz")
assert data.groupby("run_id").split.nunique().eq(1).all()
print(data.groupby(["split", "domain"]).agg(
    episodes=("run_id", "nunique"), samples=("time_s", "size")
).to_string())


                           episodes  samples
split       domain                          
calibration in_domain            12     7200
test        context_shift        28    16800
            in_domain            56    33600
            noise_shift          28    16800
train       in_domain            24    14400


## 2. Reconstruct a displayed residual score

The anomaly score is the maximum absolute standardized sensor residual. The stored file contains these components, so one displayed score can be checked without trusting the chart.

In [3]:
predictions = pd.read_csv(root / "artifacts/predictions.csv.gz")
summary = json.loads((root / "artifacts/synthetic_summary.json").read_text())
case = predictions[(predictions.scenario == "pump_wear") &
                   (predictions.domain == "in_domain") &
                   (predictions.time_s == 700)].iloc[0]
columns = [c for c in predictions if c.startswith("residual_z_")]
reconstructed = float(case[columns].max())
assert np.isclose(reconstructed, case.contextual_residual_score, rtol=1e-8)
threshold = summary["model"]["thresholds"]["contextual_residual"]
print(case[["run_id", "time_s", "contextual_residual_status"]].to_string())
print("Residual score:", round(reconstructed, 3))
print("Calibration threshold:", round(threshold, 3))
print("Score / threshold:", round(reconstructed / threshold, 3))


run_id                        test-0052
time_s                              700
contextual_residual_status        ALARM
Residual score: 54.074
Calibration threshold: 3.214
Score / threshold: 16.826


## 3. Count events rather than adjusted points

An alarm must start after the annotated onset. A detector already in alarm before a fault is not counted as signalling that new fault.

In [4]:
events = pd.read_csv(root / "artifacts/event_metrics.csv")
grouped = events.groupby(["domain", "method"]).agg(
    events=("process_event", "sum"),
    detected=("detected_event", "sum"),
    false_starts=("false_starts", "sum"),
    normal_hours=("normal_exposure_h", "sum"),
)
grouped["recall"] = grouped.detected / grouped.events
grouped["false_starts_per_hour"] = grouped.false_starts / grouped.normal_hours.replace(0, np.nan)
print(grouped.round(3).to_string())


                                   events  detected  false_starts  normal_hours  recall  false_starts_per_hour
domain        method                                                                                          
context_shift contextual_residual      12         0             0         0.000   0.000                    NaN
              isolation_forest         12         0             0         0.000   0.000                    NaN
              raw_limits               12         0             0         0.000   0.000                    NaN
in_domain     contextual_residual      24        24             1        10.286   1.000                  0.097
              isolation_forest         24        11             7        10.286   0.458                  0.681
              raw_limits               24        11            10        10.286   0.458                  0.972
noise_shift   contextual_residual      12         0            36         5.084   0.000                  7.080
 

## 4. Inspect the stress failure

Compare event recall with average precision and normal-time alarm occupancy. Good ranking of fault points cannot recover an alarm that started too early and stayed active.

In [5]:
metrics = pd.DataFrame(summary["summary"])
focus = metrics[metrics.method == "contextual_residual"]
print(focus[["domain", "event_recall", "point_average_precision",
             "normal_alarm_occupancy", "coverage"]].round(3).to_string(index=False))


       domain  event_recall  point_average_precision  normal_alarm_occupancy  coverage
    in_domain           1.0                    0.985                   0.002     0.827
  noise_shift           0.0                    0.966                   0.973     0.824
context_shift           0.0                      NaN                     NaN     0.000


## 5. Inspect the measured-data split

The UCI task uses completed cycles. Groups are full condition recipes, not patients or independent physical rigs. This distinction limits the conclusion.

In [6]:
real = json.loads((root / "artifacts/real_benchmark.json").read_text())
rp = pd.read_csv(root / "artifacts/real_benchmark_predictions.csv.gz")
groups = {s: set(g.condition_group) for s, g in rp.groupby("split")}
assert not groups["train"] & groups["test"]
assert not groups["train"] & groups["calibration"]
assert not groups["calibration"] & groups["test"]
assert not set(real["features"]) & {"cooler", "valve", "pump_leakage", "accumulator", "unstable", "cycle_id"}
print("Disjoint recipe groups:", {k: len(v) for k, v in groups.items()})
table = pd.DataFrame(real["metrics"])
print(table[table.split == "test"][["task", "model", "balanced_accuracy", "macro_f1"]].round(3).to_string(index=False))


Disjoint recipe groups: {'calibration': 27, 'test': 36, 'train': 81}
              task               model  balanced_accuracy  macro_f1
cooler_degradation         dummy_prior                0.5     0.422
cooler_degradation logistic_regression                1.0     1.000
cooler_degradation       random_forest                1.0     1.000
      pump_leakage         dummy_prior                0.5     0.313
      pump_leakage logistic_regression                1.0     1.000
      pump_leakage       random_forest                1.0     1.000


## 6. Interpretation to write yourself

Explain why the residual model's synthetic performance is favoured by its feature design. Explain why a one-rig perfect classification result is insufficient for a deployment claim. Specify a new development/test protocol before changing the alarm logic.

The first implementation was AI-assisted. Record your own experiment, configuration and interpretation in a separate commit.